In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ① Silver 테이블 읽기
df_weather = spark.table("silver.weather")
df_congest = spark.table("silver.congest")
df_road = spark.table("silver.road")
df_event = spark.table("silver.event_accident")
df_sdot = spark.table("silver.sdot")

# ② 각 테이블 최신 데이터 1개 추출
window_weather = Window.partitionBy("AREA_NM").orderBy(F.col("WEATHER_TIME").desc())
window_congest = Window.partitionBy("AREA_NM").orderBy(F.col("PPLTN_TIME").desc())
window_road = Window.partitionBy("AREA_NM").orderBy(F.col("ROAD_TRAFFIC_TIME").desc())
window_event = Window.partitionBy("AREA_NM").orderBy(F.col("collected_at").desc())

df_weather_latest = df_weather.withColumn("rn", F.row_number().over(window_weather)).filter(F.col("rn") == 1).drop("rn")
df_congest_latest = df_congest.withColumn("rn", F.row_number().over(window_congest)).filter(F.col("rn") == 1).drop("rn")
df_road_latest = df_road.withColumn("rn", F.row_number().over(window_road)).filter(F.col("rn") == 1).drop("rn")
df_event_latest = df_event.withColumn("rn", F.row_number().over(window_event)).filter(F.col("rn") == 1).drop("rn")

# ③ AREA_NM 기준 조인
df_gold = df_weather_latest.select(
    "AREA_NM", "TEMP", "SENSIBLE_TEMP", "HUMIDITY",
    "WIND_SPD", "WIND_DIRCT", "PRECIPITATION", "PRECPT_TYPE",
    "PCP_MSG", "UV_INDEX", "UV_INDEX_LVL",
    "PM10", "PM10_INDEX", "PM25", "PM25_INDEX",
    "AIR_IDX", "AIR_MSG", "WEATHER_TIME"
).join(
    df_congest_latest.select("AREA_NM", "AREA_CONGEST_LVL", "AREA_CONGEST_MSG", "AREA_PPLTN_MIN", "AREA_PPLTN_MAX", "PPLTN_TIME"),
    on="AREA_NM", how="left"
).join(
    df_road_latest.select("AREA_NM", "ROAD_TRAFFIC_SPD", "ROAD_TRAFFIC_IDX", "ROAD_TRAFFIC_TIME"),
    on="AREA_NM", how="left"
).join(
    df_event_latest.select("AREA_NM", "ACDNT_CNT", "EVENT_YN", "DST_MSG_CNT"),
    on="AREA_NM", how="left"
)

# ④ AREA_NM → 자치구 매핑
AREA_DISTRICT_MAP = {
    "가로수길": "Gangnam-gu", "강남 MICE 관광특구": "Gangnam-gu",
    "강남역": "Gangnam-gu", "고속터미널역": "Gangnam-gu",
    "교대역": "Gangnam-gu", "선릉역": "Gangnam-gu",
    "신논현역·논현역": "Gangnam-gu", "양재역": "Gangnam-gu",
    "역삼역": "Gangnam-gu", "압구정로데오거리": "Gangnam-gu",
    "청담동 명품거리": "Gangnam-gu",
    "잠실 관광특구": "Songpa-gu", "장지역": "Songpa-gu",
    "잠실종합운동장": "Songpa-gu", "잠실한강공원": "Songpa-gu",
    "잠실새내역": "Songpa-gu", "잠실역": "Songpa-gu",
    "잠실롯데타워·석촌호수": "Songpa-gu", "송리단길·호수단길": "Songpa-gu",
    "서울 암사동 유적": "Gangdong-gu",
    "고덕역": "Gangdong-gu", "천호역": "Gangdong-gu",
}

mapping_expr = F.create_map(
    *[item for pair in [(F.lit(k), F.lit(v)) for k, v in AREA_DISTRICT_MAP.items()] for item in pair]
)

# ⑤ S-DoT 구 단위 평균 집계 후 조인
df_sdot_agg = df_sdot.groupBy("district").agg(
    F.avg("avg_noise").alias("sdot_avg_noise"),
    F.avg("max_noise").alias("sdot_max_noise"),
    F.avg("avg_vibr_x").alias("sdot_avg_vibr_x"),
    F.avg("avg_vibr_y").alias("sdot_avg_vibr_y"),
    F.avg("avg_vibr_z").alias("sdot_avg_vibr_z"),
)

df_gold = df_gold.withColumn("district", mapping_expr[F.col("AREA_NM")]) \
    .join(df_sdot_agg, on="district", how="left")

# ⑥ Gold Delta Table 저장
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.walk_environment")

print("✅ Gold 저장 완료! 🐾")

✅ Gold 저장 완료! 🐾


In [0]:
# ④ Gold Delta Table로 저장
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.walk_environment")

print("✅ Gold 저장 완료! 🐾")

✅ Gold 저장 완료! 🐾


In [0]:
display(spark.sql("SELECT AREA_NM, district, sdot_avg_noise, sdot_avg_vibr_z FROM gold.walk_environment LIMIT 5"))

AREA_NM,district,sdot_avg_noise,sdot_avg_vibr_z
가로수길,Gangnam-gu,50.3,0.72
강남 MICE 관광특구,Gangnam-gu,50.3,0.72
강남역,Gangnam-gu,50.3,0.72
고덕역,Gangdong-gu,55.46478873239437,0.5640845070422534
고속터미널역,Gangnam-gu,50.3,0.72
